In [1]:
import json

from datasets import load_dataset
from openai import OpenAI

from transformers import AutoTokenizer

/opt/conda/envs/gen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = load_dataset("slm-research-vn/ultrachat_200k")

In [3]:
messages_column_name = "messages"
lst_messages = data["train"][messages_column_name]

In [4]:
lst_instructions = data["train"]["instruction"]

In [5]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer.chat_template = "{{- bos_token }}{%- for message in messages %}{{- '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' }}{%- endfor %}{%- if add_generation_prompt %}{{- '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{%- endif %}"

In [7]:
api_key="EMPTY"
base_url="http://localhost:1234/v1"
model = "Meta-Llama-3.1-8B-Instruct"

client = OpenAI(api_key=api_key, base_url=base_url)
tmp_messages = [
    {"role": "user", "content": "Hello, how are you?"}, 
    {"role": "assistant", "content": "I'm fine, thank you."}, 
    {"role": "user", "content": "Explain to me the concept of Optimal Transport in one sentence."},
    {"role": "assistant", "content": "Optimal Transport is a mathematical framework that allows for the comparison of two probability distributions."}
]

print(tokenizer.apply_chat_template(tmp_messages, tokenize=False))


def api_chat_generate(messages, sampling_params=None):
    if not sampling_params:
        sampling_params = {
            "temperature": 0.8,
            "max_tokens": 512,
            "top_p": 1.0,
            "stop": "<|eot_id|>",
        }
    
    response = client.chat.completions.create(model=model, messages=messages, **sampling_params)
    
    return response.choices[0].message.content

def api_generate(prompt, sampling_params=None):
    if not sampling_params:
        sampling_params = {
            "temperature": 0.8,
            "max_tokens": 512,
            "top_p": 1.0,
            "stop": "<|eot_id|>",
        }
    
    response = client.completions.create(model=model, prompt=prompt, **sampling_params)
    
    return response.choices[0].text

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello, how are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I'm fine, thank you.<|eot_id|><|start_header_id|>user<|end_header_id|>

Explain to me the concept of Optimal Transport in one sentence.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Optimal Transport is a mathematical framework that allows for the comparison of two probability distributions.<|eot_id|>


In [9]:
def generate_user_message(messages) -> str:
    assert messages[-1]["role"] == "assistant"
    
    # reverse the role user -> assistant; assistant -> user
    reversed_messages = [{"role": "assistant" if m["role"] == "user" else "user", "content": m["content"]} for m in messages]
    
    assert reversed_messages[-1]["role"] == "user"
    
    print(reversed_messages)
    
    system_prompt = """You are a curious chatbot designed to ask insightful questions or provide high-quality instructions based on the user's message. Ensure that the conversation with the user flows naturally and smoothly."""
    
    return api_chat_generate([{"role": "system", "content": system_prompt}] + reversed_messages)


generate_user_message(tmp_messages)

[{'role': 'assistant', 'content': 'Hello, how are you?'}, {'role': 'user', 'content': "I'm fine, thank you."}, {'role': 'assistant', 'content': 'Explain to me the concept of Optimal Transport in one sentence.'}, {'role': 'user', 'content': 'Optimal Transport is a mathematical framework that allows for the comparison of two probability distributions.'}]


'Comparison" is a broad term - what specific aspects of these probability distributions can Optimal Transport help compare, and what kinds of insights can it provide?'

In [8]:
def is_related(current_messages, next_message):
    conversation = "\n".join([f"{m['role']}: {m['content']}" for m in current_messages])
    message = f"{next_message['role']}: {next_message['content']}"

    prompt = f'''Given the following conversation:
### Conversation
```
{conversation}
```

Can we continue the conversation with the following message without breaking the flow of the conversation?
### Message
```
{message}
```

### Output Format
Begin your evaluation by providing a short explanation why the message can or cannot be continued in the conversation. Then, provide the output as either "YES" or "NO".

Please provide your answer in the following format:
```
{{
    "explanation": "Your explanation here.",
    "output": "YES" or "NO"
}}
```
'''
    fmt_prompt = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True) + "{"
    
    response = api_generate(fmt_prompt, sampling_params={"temperature": 0.0, "max_tokens": 512, "stop": ["}"]})
    
    response = "{" + response + "}"
    
    try:
        response = json.loads(response)
        
        if response["output"] == "YES":
            return True
        return False
    
    except json.JSONDecodeError:
        print(f"Error when decoding response: {response}")
        return False

def synthesize(instruction, reference_messages):
    synthesized_messages = [
        {"role": "user", "content": instruction}
    ]
    
    while True:
        response = api_chat_generate(synthesized_messages) # call the API to generate the next message
        synthesized_messages.append({"role": "assistant", "content": response})
        if len(synthesized_messages) == len(reference_messages):
            break
        
        next_message = reference_messages[len(synthesized_messages)]
        
        if is_related(synthesized_messages, next_message): # call the API to check if the next message is related to the conversation
            synthesized_messages.append(next_message)
        
        else:
            print("The conversation has ended.")
            break
        
    return synthesized_messages

In [9]:
synthesized_messages = synthesize(lst_instructions[0], lst_messages[0])

In [11]:
import concurrent.futures


with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = [
        executor.submit(synthesize, lst_instructions[i], lst_messages[i])
        for i in range(10)
    ]
    results = [future.result() for future in futures]

In [12]:
results[0]

[{'role': 'user',
  'content': 'What are the common features of stealth bombers, and how do they differ from traditional bombers?'},
 {'role': 'assistant',
  'content': 'Stealth bombers" is a bit of a misnomer, as most modern stealth bombers are actually "low-observable" aircraft designed to reduce their visibility to radar and other detection methods, rather than being completely stealthy. That being said, here are the common features of stealth bombers and how they differ from traditional bombers:\n\n**Common features of stealth bombers:**\n\n1. **Radome-shaping**: Stealth aircraft have a unique, curved shape that helps to dissipate radar waves and reduce their detectability.\n2. **Materials**: Stealth aircraft are often made with radar-absorbing materials (RAMs) that absorb or scatter radar waves, rather than reflecting them.\n3. **Flat, smooth surfaces**: Stealth aircraft have reduced angles and edges, which minimize radar reflectivity.\n4. **Radome coverage**: Stealth aircraft oft

In [10]:
x = [1,2,3]

def test(x):
    x.append(4)
    return x

y = test(x)

x

[1, 2, 3, 4]